[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsinghlab/pyaging/blob/main/tutorials/tutorial_cpgptgrimage3.ipynb) [![Open In nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/rsinghlab/pyaging/blob/main/tutorials/tutorial_cpgptgrimage3.ipynb)

# Applying CpGPT to GSE185307 - Nanopore data

This notebook applies the "Cancer" fine-tuned model to the Oxford Nanopore data from GSE185307.



## 1. Setup Environment

We'll import the necessary Python packages and set up our environment for CpGPT. We'll be using a mix of standard data science libraries and CpGPT-specific modules. We'll also set some important variables that will be used throughout the notebook. Pay attention to these as you may need to adjust them based on your specific setup and requirements.

In [2]:
# Random seed for reproducibility
RANDOM_SEED = 42

# Directory paths
DEPENDENCIES_DIR = "CpGPT/dependencies"
LLM_DEPENDENCIES_DIR = DEPENDENCIES_DIR + "/human"
DATA_DIR = "CpGPT/data"
PROCESSED_DIR = "CpGPT/data/tutorials/processed/quick_setup"

MODEL_NAME = "cancer"
MODEL_CHECKPOINT_PATH = f"CpGPT/dependencies/model/weights/{MODEL_NAME}.ckpt"
MODEL_CONFIG_PATH = f"CpGPT/dependencies/model/config/{MODEL_NAME}.yaml"
MODEL_VOCAB_PATH = f"CpGPT/dependencies/model/vocab/{MODEL_NAME}.json"

import os
data_dir = "CpGPT/data/GSE185307_RAW/"
output_file = os.path.join(data_dir, "GSE185307.arrow")
ARROW_DF_PATH = "CpGPT/data/cpgcorpus/raw/GSE182215/GPL13534/betas/QCDPB.arrow"
ARROW_DF_PATH = output_file

ARROW_DF_FILTERED_PATH = "CpGPT/data/tutorials/raw/toy_filtered.arrow"

# The maximum context length to give to the model
MAX_INPUT_LENGTH = 20_000 # you might wanna go higher hardware permitting
MAX_ATTN_LENGTH = 1_000

> **⚠️ Warning**
> 
> It is recommended to have a GPU for inference as CPU might be slow.
> 
> Reconstructing the methylome for a few hundred samples might take up to one hour on a CPU. ⌛
>
> This might be a great exercise in testing your patience.

### 1.2 Import packages


In [3]:
# Standard library imports
import warnings
import os
import json

warnings.simplefilter(action="ignore", category=FutureWarning)

# Plotting imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyaging as pya
import seaborn as sns

# Lightning imports
from lightning.fabric.utilities.seed import seed_everything

# cpgpt-specific imports
from cpgpt.data.components.cpgpt_datasaver import CpGPTDataSaver
from cpgpt.data.cpgpt_datamodule import CpGPTDataModule
from cpgpt.trainer.cpgpt_trainer import CpGPTTrainer
from cpgpt.data.components.dna_llm_embedder import DNALLMEmbedder
from cpgpt.data.components.illumina_methylation_prober import IlluminaMethylationProber
from cpgpt.infer.cpgpt_inferencer import CpGPTInferencer
from cpgpt.model.cpgpt_module import m_to_beta

# Set random seed for reproducibility
seed_everything(RANDOM_SEED, workers=True)

Seed set to 42


42

## 2. Retrieve DNA LLM Embeddings

To retrieve the DNA LLM Embeddings, there are two options:
- download the dependencies with all of the sequence embeddings for the CpG sites targeted by the Illumina arrays;
- generate from scratch using the DNA LLM directly for loci outside of the ones already available for download.

In [4]:
# First let's declare the inferencer
inferencer = CpGPTInferencer(dependencies_dir=DEPENDENCIES_DIR, data_dir=DATA_DIR)

cpgpt -CpGPTInferencer: Initializing class CpGPTInferencer.
cpgpt -CpGPTInferencer: Using device: cpu.
cpgpt -CpGPTInferencer: Using dependencies directory: CpGPT/dependencies
cpgpt -CpGPTInferencer: Using data directory: CpGPT/data
cpgpt -CpGPTInferencer: There are 19 CpGPT models available such as age, age_cot, average_adultweight, etc.
cpgpt -CpGPTInferencer: There are 2089 GSE datasets available such as GSE100184, GSE100208, GSE100209, etc.


### 2.1 Download Dependencies


The already-processed dependencies contain the sequence embeddings for both human (`s3://cpgpt-lucascamillo-public/dependencies/human`) and several mammalian species (`s3://cpgpt-lucascamillo-public/dependencies/mammalian`). Here, let's use the human as an example:

In [5]:
inferencer.download_dependencies(species="human")

cpgpt -CpGPTInferencer: Dependencies for human already exist at CpGPT\dependencies\human (skipping download).


### 2.2 Generate DNA LLM Embeddings


To generate genomic embeddings for loci outside of the ones already available for download, we can use the `DNALLMEmbedder` class. We need the loci in a list with the following format from ENSEMBL: 'chromosome:position'. Be mindful as this function can take a long time to run dependending on your GPU. For instance, embeddings ~1M genomic loci from the Illumina arrays takes about 12h in an RTX 4090.

In [6]:
if not os.path.exists(LLM_DEPENDENCIES_DIR):

    # List CpG genomic locations
    example_genomic_locations = ['1:100000', '1:250500', 'X:2031253']

    # Declare required class
    embedder = DNALLMEmbedder(dependencies_dir=LLM_DEPENDENCIES_DIR)

    # Parse the embeddings
    embedder.parse_dna_embeddings(
        example_genomic_locations,
        "homo_sapiens",
        dna_llm="nucleotide-transformer-v2-500m-multi-species",
        dna_context_len=2001,
    )

## 3. Download and Load Model

Please first check the model zoo for the available models and their corresponding features on the README.md file. To load any given model, you first need to define the dictionary structure with the hyperparameters and use the `CpGPTInferencer` class.

### 3.1 Download Checkpoint and Configuration Files

In [7]:
# Download the checkpoint and configuration files
inferencer.download_model(MODEL_NAME)

cpgpt -CpGPTInferencer: Model checkpoint already exists at CpGPT/dependencies/model/weights/cancer.ckpt (skipping download).
cpgpt -CpGPTInferencer: Model config already exists at CpGPT/dependencies/model/config/cancer.yaml (skipping download).
cpgpt -CpGPTInferencer: Model vocabulary already exists at CpGPT/dependencies/model/vocab/cancer.json (skipping download).
cpgpt -CpGPTInferencer: Successfully downloaded model 'cancer'.


### 3.2 Load Model

In [8]:
# Load the model configuration
config = inferencer.load_cpgpt_config(MODEL_CONFIG_PATH)

# Load the model weights
model = inferencer.load_cpgpt_model(
    config,
    model_ckpt_path=MODEL_CHECKPOINT_PATH,
    strict_load=True,
)

cpgpt -CpGPTInferencer: Loaded CpGPT model config.
cpgpt -CpGPTInferencer: Instantiated CpGPT model from config.
cpgpt -CpGPTInferencer: Using device: cpu.
cpgpt -CpGPTInferencer: Loading checkpoint from: CpGPT/dependencies/model/weights/cancer.ckpt
cpgpt -CpGPTInferencer: Checkpoint loaded into the model.


## 4 Prepare Data Objects

In order to perform inference, we need to prepare the data objects, which are essentially memory-mapped versions for faster loading. As an example, let's download a toy dataset from the _CpGCorpus_ database.

### 4.1 Download and Load Toy Data

### Create arrow file for GSE185307, if it does not exists yet

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
from glob import glob
import os

# Relative paths from project root
# data_dir = "CpGPT/data/GSE185307_RAW/"
# output_file = os.path.join(data_dir, "GSE185307.feather") # Defined at beginning of notebook


if os.path.exists(output_file):
    print(f"[INFO] Feather file already exists at {output_file}, skipping generation.")
else:
    print(f"[INFO] Generating feather file from bedgraph files in: {data_dir}")
    
    files = glob(os.path.join(data_dir, "*.bedgraph.gz"))[0:3]
    print(files)
    sample_dfs = []

    for file in files:
        print(file)
        df = pd.read_csv(file, sep="\t", compression="gzip", header=None,
                         names=["Chrom", "start", "end", "sample", "beta"])
        df["cpg_id"] = df["Chrom"] + "_" + df["start"].astype(str)
        sample_name = os.path.basename(file).split("_")[0] # Extract sample name from filename
        sample_series = df.set_index("cpg_id")["beta"]
        sample_series.name = sample_name
        sample_dfs.append(sample_series)


    print(f"[INFO] Feather file saved to {output_file}")


[INFO] Generating feather file from bedgraph files in: CpGPT/data/GSE185307_RAW/
['CpGPT/data/GSE185307_RAW\\GSM6069339_HU005.10.hg38.sorted.grouped.0based.bedgraph.gz', 'CpGPT/data/GSE185307_RAW\\GSM6069340_HU005.11.hg38.sorted.grouped.0based.bedgraph.gz', 'CpGPT/data/GSE185307_RAW\\GSM6069341_HU005.12.hg38.sorted.grouped.0based.bedgraph.gz']
CpGPT/data/GSE185307_RAW\GSM6069339_HU005.10.hg38.sorted.grouped.0based.bedgraph.gz
CpGPT/data/GSE185307_RAW\GSM6069340_HU005.11.hg38.sorted.grouped.0based.bedgraph.gz
CpGPT/data/GSE185307_RAW\GSM6069341_HU005.12.hg38.sorted.grouped.0based.bedgraph.gz


AttributeError: module 'pyarrow.feather' has no attribute 'write_arrow'

In [ ]:
if os.path.exists(output_file):
    print(f"[INFO] Feather file already exists at {output_file}, skipping generation.")
else:
    methylation_df = pd.concat(sample_dfs, axis=1).transpose()
    methylation_df.index.names = ["GSM_ID"]

    
    arrow_table = pa.Table.from_pandas(methylation_df)
    feather.write_arrow(arrow_table, output_file)
    methylation_df.rename(columns={"GSM_ID": "cpg_id"})


KeyboardInterrupt: 

In [ ]:
if os.path.exists(output_file):
    print(f"[INFO] Feather file already exists at {output_file}, skipping generation.")
else:
    methylation_df.rename(columns={"GSM_ID": "cpg_id"})


In [ ]:
import csv

def convert_df_columns_to_cgids_by_reassignment(df: pd.DataFrame, manifest_filepath: str) -> pd.DataFrame:
    """
    Efficiently converts DataFrame column names from 'chrX_Y' to 'cgID' using streamed manifest.
    Replaces column names in-place and prints mapping statistics.

    Args:
        df (pd.DataFrame): Wide DataFrame with coordinates as column names.
        manifest_filepath (str): Path to Illumina EPIC manifest file (CSV).

    Returns:
        pd.DataFrame: Same DataFrame with updated column names.
    """
    print(f"[INFO] Streaming manifest from: {manifest_filepath}")
    
    coord_to_cgid_map = {}
    with open(manifest_filepath, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            try:
                cpg_id = row['IlmnID']
                chr_val = row['CHR']
                pos = row['MAPINFO']
                if chr_val and pos:
                    coord = f"chr{chr_val}_{pos}"
                    coord_to_cgid_map[coord] = cpg_id
            except KeyError:
                continue

    print(f"[INFO] Mapping table created with {len(coord_to_cgid_map)} entries")

    # Perform mapping
    original_columns = list(df.columns)
    new_columns = [coord_to_cgid_map.get(col, col) for col in original_columns]
    df.columns = new_columns

    # Compute mapping stats
    total = len(original_columns)
    mapped = sum(1 for col in original_columns if col in coord_to_cgid_map)
    unmapped = total - mapped
    pct = mapped / total * 100

    print(f"[STATS] Total CpGs in DataFrame: {total}")
    print(f"[STATS] Mapped to cgIDs: {mapped} ({pct:.2f}%)")
    print(f"[STATS] Unmapped CpGs: {unmapped}")

    return df
methylation_df_cg = convert_df_columns_to_cgids_by_reassignment(methylation_df, manifest_path)

[INFO] Streaming manifest from: CpGPT/data/GSE185307_RAW/EPIC-8v2-0_A1.csv


ParserError: Error tokenizing data. C error: out of memory

In [58]:
import csv

def convert_df_columns_to_cgids(df: pd.DataFrame, manifest_filepath: str, chunk_size: int = 10000) -> pd.DataFrame:
    """
    Memory-efficient conversion of DataFrame column names from 'chrX_Y' to 'cgID' using streamed manifest.
    Renames columns in chunks to avoid full DataFrame copy.

    Args:
        df (pd.DataFrame): Wide DataFrame with coordinates as column names.
        manifest_filepath (str): Path to the Illumina EPIC manifest file (CSV).
        chunk_size (int): Number of columns to rename at a time (default: 10,000).

    Returns:
        pd.DataFrame: Same DataFrame with updated column names (in-place).
    """
    print(f"[INFO] Streaming manifest from: {manifest_filepath}")
    
    coord_to_cgid_map = {}

    with open(manifest_filepath, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            try:
                cpg_id = row['IlmnID']
                chr_val = row['CHR']
                pos = row['MAPINFO']
                if chr_val and pos:
                    coord = f"chr{chr_val}_{pos}"
                    coord_to_cgid_map[coord] = cpg_id
            except KeyError:
                continue

    print(f"[INFO] Mapping table created with {len(coord_to_cgid_map)} entries")

    # Chunked column renaming
    all_columns = list(df.columns)
    for i in range(0, len(all_columns), chunk_size):
        chunk = all_columns[i:i + chunk_size]
        rename_map = {col: coord_to_cgid_map[col] for col in chunk if col in coord_to_cgid_map}
        df.rename(columns=rename_map, inplace=True)
        print(f"[INFO] Renamed columns {i} to {i + len(chunk)}")

    print("[INFO] All columns renamed (where mapping was found)")
    return df


manifest_path = "CpGPT/data/GSE185307_RAW/EPIC-8v2-0_A1.csv"
manifest_path = "CpGPT/data/EPIC-8v2-0_A1.csv"
manifest_path = r"CpGPT/data/GSE185307_RAW/EPIC-8v2-0_A1.csv"


methylation_df_cg = convert_df_columns_to_cgids(methylation_df, manifest_path)

[INFO] Streaming manifest from: CpGPT/data/GSE185307_RAW/EPIC-8v2-0_A1.csv
[INFO] Mapping table created with 0 entries
[INFO] Renamed columns 0 to 10000
[INFO] Renamed columns 10000 to 20000
[INFO] Renamed columns 20000 to 30000
[INFO] Renamed columns 30000 to 40000
[INFO] Renamed columns 40000 to 50000
[INFO] Renamed columns 50000 to 60000
[INFO] Renamed columns 60000 to 70000
[INFO] Renamed columns 70000 to 80000
[INFO] Renamed columns 80000 to 90000
[INFO] Renamed columns 90000 to 100000
[INFO] Renamed columns 100000 to 110000
[INFO] Renamed columns 110000 to 120000
[INFO] Renamed columns 120000 to 130000
[INFO] Renamed columns 130000 to 140000
[INFO] Renamed columns 140000 to 150000
[INFO] Renamed columns 150000 to 160000
[INFO] Renamed columns 160000 to 170000
[INFO] Renamed columns 170000 to 180000
[INFO] Renamed columns 180000 to 190000


KeyboardInterrupt: 

In [ ]:
methylation_df.columns

cpg_id,chr1_10635,chr1_10637,chr1_10640,chr1_10643,chr1_10649,chr1_10659,chr1_10661,chr1_10664,chr1_10666,chr1_10669,...,chrY_57210721,chrY_57210737,chrY_57210752,chrY_57210797,chrY_57210813,chrY_57212067,chrY_57212102,chrY_57212119,chrY_57212153,chrY_57212397
GSM_ID,,,,,,,,,,,,,,,,,,,,,
GSM6069339,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSM6069340,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSM6069341,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# inferencer.download_cpgcorpus_dataset("GSE182215")


cpgpt -CpGPTInferencer: Dataset GSE182215 already exists at CpGPT\data\cpgcorpus\raw\GSE182215 (skipping download).


In [ ]:
# DIsease dataset
#inferencer.download_cpgcorpus_dataset("gse213478")

There is no need to impute the methylation data for CpGPT -- it simply ignores the missing values.

In [ ]:
df = pd.read_feather(ARROW_DF_PATH)
df.set_index('GSM_ID', inplace=True)
df.head()

,cg00000029,cg00000108,cg00000109,cg00000165,cg00000236,cg00000289,cg00000292,cg00000321,cg00000363,cg00000622,...,rs7746156,rs798149,rs845016,rs877309,rs9292570,rs9363764,rs939290,rs951295,rs966367,rs9839873
GSM_ID,,,,,,,,,,,,,,,,,,,,,
GSM5525203,0.324878,0.958127,0.853575,NaN,0.854126,NaN,0.791321,0.249139,0.291044,0.013388,...,0.025733,0.462424,0.100629,0.979795,0.019506,0.947983,0.054746,0.495169,0.675477,0.782523
GSM5525204,0.298781,0.939107,0.897159,0.217733,0.874908,NaN,0.828892,0.228412,0.397047,0.013511,...,0.978182,0.017012,0.930166,0.568672,0.493534,0.042713,0.620214,0.511735,0.069878,0.941439
GSM5525205,NaN,NaN,NaN,NaN,0.472385,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSM5525206,0.125208,0.961126,0.822223,0.229362,0.861563,NaN,0.877324,0.178019,0.377428,0.017229,...,0.550734,0.444213,0.644010,0.563410,0.460727,0.961204,0.051867,0.487339,0.126543,0.899515
GSM5525207,0.278861,0.970059,0.929905,0.171255,0.907603,0.820531,0.893471,0.185116,0.377647,0.012591,...,0.495878,0.020168,0.483792,0.021345,0.015444,0.968064,0.551504,0.970979,0.062037,0.764114


### 4.2 Filter Vocab Features and Save Data

While not strictly required, filtering for the features used in finetuning gives you the best chance of achieving good performance.

In [ ]:
# Load list
vocab = json.load(open(MODEL_VOCAB_PATH, 'r'))

In [ ]:
df = df.loc[:, df.columns.isin(vocab['input'])].dropna(thresh=10)
df.head()

,cg00000292,cg00002426,cg00003994,cg00005847,cg00008493,cg00009407,cg00011459,cg00012199,cg00012386,cg00012792,...,cg27650175,cg27650434,cg27652350,cg27653134,cg27654142,cg27655905,cg27657283,cg27662379,cg27662877,cg27665659
GSM_ID,,,,,,,,,,,,,,,,,,,,,
GSM5525203,0.791321,0.905377,0.091164,0.090651,0.951774,0.048334,0.938640,0.035517,0.056877,0.087570,...,0.045768,0.072923,0.132974,0.949820,0.065028,0.063921,0.052416,0.077480,0.043270,0.057590
GSM5525204,0.828892,0.964620,0.042569,0.125086,0.945982,0.052446,0.951808,0.052040,0.063641,0.102222,...,0.053291,0.089544,0.130468,0.959386,0.066889,0.055794,0.044713,0.069569,0.039361,0.070515
GSM5525205,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.058280,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSM5525206,0.877324,0.920315,0.042593,0.201401,0.951397,0.058045,0.947452,0.051123,0.060039,0.091149,...,0.046296,0.112071,0.096274,0.927876,0.088435,0.056817,0.050824,0.068512,0.070187,0.063018
GSM5525207,0.893471,0.943485,0.039004,0.139082,0.952827,0.045738,0.950823,0.036847,0.041898,0.082241,...,0.047588,0.105776,0.114468,0.940735,0.057654,0.045630,0.036157,0.051430,0.041078,0.082783


In [ ]:
df.to_feather(ARROW_DF_FILTERED_PATH)

### 4.3 Memory-Map Data

In order to perform inference, we need to memory-map the data. This is done by using the `CpGPTDataSaver` class. We first need to define the `DNALLMEmbedder` and `IlluminaMethylationProber` classes, which contain the information about the DNA LLM Embeddings and the conversion between Illumina array probes to genomic locations, respectively.

In [ ]:
embedder = DNALLMEmbedder(dependencies_dir=LLM_DEPENDENCIES_DIR)

cpgpt -DNALLMEmbedder: Initializing class DNALLMEmbedder.
cpgpt -DNALLMEmbedder: Genome files will be stored under CpGPT\dependencies\human\genomes.
cpgpt -DNALLMEmbedder: DNA embeddings will be stored under CpGPT\dependencies\human\dna_embeddings and subdirectories.
cpgpt -DNALLMEmbedder: Ensembl metadata dictionary loaded successfully


In [ ]:
prober = IlluminaMethylationProber(dependencies_dir=LLM_DEPENDENCIES_DIR, embedder=embedder)

cpgpt -IlluminaMethylationProber: Initializing class IlluminaMethylationProber.
cpgpt -IlluminaMethylationProber: Illumina methylation manifest files will be stored under CpGPT\dependencies\human\manifests.
cpgpt -IlluminaMethylationProber: Illumina metadata dictionary loaded successfully.


In [ ]:
# Define datasaver  
quick_setup_datasaver = CpGPTDataSaver(data_paths=ARROW_DF_FILTERED_PATH, processed_dir=PROCESSED_DIR)



cpgpt -CpGPTDataSaver: Initializing class CpGPTDataSaver.
cpgpt -CpGPTDataSaver: Dataset folders will be stored under CpGPT/data/tutorials/processed/quick_setup.
cpgpt -CpGPTDataSaver: Loaded existing dataset metrics.
cpgpt -CpGPTDataSaver: Loaded existing genomic locations.


In [ ]:
# Process the file
quick_setup_datasaver.process_files(prober, embedder)

cpgpt -CpGPTDataSaver: Starting file processing.
cpgpt -CpGPTDataSaver: 1 files already processed. Skipping those.


### 4.4 Declare data module

Let's define two data modules: one for the forward pass and reconstructing the methylation, and another one the attention weights.

In [ ]:
# Define datamodule
quick_setup_datamodule = CpGPTDataModule(
    predict_dir=PROCESSED_DIR,
    dependencies_dir=LLM_DEPENDENCIES_DIR,
    batch_size=1,
    num_workers=0,
    max_length=MAX_INPUT_LENGTH,
    dna_llm=config.data.dna_llm,
    dna_context_len=config.data.dna_context_len,
    sorting_strategy=config.data.sorting_strategy,
    pin_memory=False,
)

# Define datamodule
quick_setup_datamodule_attn = CpGPTDataModule(
    predict_dir=PROCESSED_DIR,
    dependencies_dir=LLM_DEPENDENCIES_DIR,
    batch_size=1,
    num_workers=0,
    max_length=MAX_ATTN_LENGTH,
    dna_llm=config.data.dna_llm,
    dna_context_len=config.data.dna_context_len,
    sorting_strategy=config.data.sorting_strategy,
    pin_memory=False,
)

cpgpt -DNALLMEmbedder: Initializing class DNALLMEmbedder.
cpgpt -DNALLMEmbedder: Genome files will be stored under CpGPT\dependencies\human\genomes.
cpgpt -DNALLMEmbedder: DNA embeddings will be stored under CpGPT\dependencies\human\dna_embeddings and subdirectories.
cpgpt -DNALLMEmbedder: Ensembl metadata dictionary loaded successfully
cpgpt -DNALLMEmbedder: Initializing class DNALLMEmbedder.
cpgpt -DNALLMEmbedder: Genome files will be stored under CpGPT\dependencies\human\genomes.
cpgpt -DNALLMEmbedder: DNA embeddings will be stored under CpGPT\dependencies\human\dna_embeddings and subdirectories.
cpgpt -DNALLMEmbedder: Ensembl metadata dictionary loaded successfully


## 5. Run Inference

There are several ways to perform inference with CpGPT. Here, we'll go through the most common ones.

### 5.1 Declare Trainer

Given all models were trained under mixed precision, we'll use the `precision="16-mixed"` argument. However, if you finetune it using a different precision, you can change that accordingly.

In [ ]:
trainer = CpGPTTrainer(precision="16-mixed")

c:\Users\dallo\workspace\cancer-methylation-detection\.conda\Lib\site-packages\lightning\pytorch\trainer\connectors\accelerator_connector.py:513: You passed `Trainer(accelerator='cpu', precision='16-mixed')` but AMP with fp16 is not supported on CPU. Using `precision='bf16-mixed'` instead.
Using bfloat16 Automatic Mixed Precision (AMP)
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\dallo\workspace\cancer-methylation-detection\.conda\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the def

### 5.2 Get Sample Embeddings

In [ ]:
quick_setup_sample_embeddings = trainer.predict(
    model=model,
    datamodule=quick_setup_datamodule,
    predict_mode="forward",
    return_keys=["sample_embedding"]
)

cpgpt -CpGPTDataset: Initializing class CpGPTDataset.
cpgpt -CpGPTDataset: Loaded existing dataset metrics.


Output()

c:\Users\dallo\workspace\cancer-methylation-detection\.conda\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


In [ ]:
quick_setup_sample_embeddings

{'sample_embedding': tensor([[-1.7904e-01, -4.6749e-02, -7.3175e-02,  ..., -3.5239e-02,
          -4.3050e-02, -9.6619e-02],
         [-2.1246e-01, -8.2881e-02, -8.7091e-02,  ..., -2.5961e-02,
          -5.6928e-02, -9.6733e-02],
         [-7.8577e-02, -3.5787e-02, -8.6237e-02,  ..., -4.9353e-02,
          -5.8711e-05, -7.7660e-02],
         ...,
         [-2.6589e-01, -7.5843e-02, -2.8284e-02,  ..., -9.6518e-02,
          -4.1982e-02, -2.0806e-02],
         [-3.0264e-01, -7.2444e-02, -6.3257e-02,  ..., -1.2485e-01,
          -4.6133e-02,  5.0521e-04],
         [-2.5430e-01, -6.7684e-02, -2.2028e-02,  ..., -8.5776e-02,
          -5.7706e-02, -8.8427e-03]])}

### 5.3 Predict Phenotypes

In [ ]:
quick_setup_pred_conditions = trainer.predict(
    model=model,
    datamodule=quick_setup_datamodule,
    predict_mode="forward",
    return_keys=["pred_conditions"]
)

cpgpt -CpGPTDataset: Initializing class CpGPTDataset.
cpgpt -CpGPTDataset: Loaded existing dataset metrics.


Output()

In [ ]:
quick_setup_pred_conditions

{'pred_conditions': tensor([[1.3984],
         [1.9375],
         [4.5312],
         [2.0000],
         [1.9766],
         [1.5781],
         [2.0469],
         [1.5469],
         [1.7500],
         [2.4062],
         [1.7109],
         [1.1172],
         [1.7969],
         [2.0469],
         [1.6719],
         [1.9219],
         [1.4844],
         [1.7344],
         [1.9844],
         [2.0000],
         [2.3125],
         [2.0000],
         [2.0000],
         [2.1094],
         [1.9688],
         [2.2188],
         [2.1406],
         [1.8750],
         [1.8047],
         [2.2812],
         [1.8516],
         [1.2500],
         [1.6797],
         [2.2969],
         [1.7734],
         [1.9141],
         [1.7188],
         [1.2422],
         [3.3594],
         [3.2188],
         [3.4688],
         [3.0938],
         [3.3906]], dtype=torch.bfloat16)}

### 5.4 Reconstruct Methylation

As an example, let's get some the reconstructed methylation values for some locations of interest based on the Illumina probes.

In [ ]:
# Random probes for demonstration
probes = list(df.columns[0:100])

probes[0:5]

['cg00000292', 'cg00002426', 'cg00003994', 'cg00005847', 'cg00008493']

In [ ]:
# Convert probes to genomic locations
genomic_locations = prober.locate_probes(probes, "homo_sapiens")

genomic_locations[0:5]

['16:28878778', '3:57757815', '7:15686236', '2:176164344', '14:93347430']

In [ ]:
quick_setup_pred_meth = trainer.predict(
    model=model,
    datamodule=quick_setup_datamodule,
    predict_mode="reconstruct",
    genomic_locations=genomic_locations,
    species="homo_sapiens",
    return_keys=["pred_meth"],
)

cpgpt -CpGPTDataset: Initializing class CpGPTDataset.
cpgpt -CpGPTDataset: Loaded existing dataset metrics.


Output()

Be mindful as the reconstructed values are M values, not beta values. Therefore, you need to convert them to beta values using the `m_to_beta` function.

In [ ]:
quick_setup_pred_meth["pred_meth"] = m_to_beta(quick_setup_pred_meth["pred_meth"])
quick_setup_pred_meth

{'pred_meth': tensor([[0.9453, 0.9336, 0.0688,  ..., 0.0488, 0.9766, 0.7188],
         [0.9531, 0.9414, 0.0708,  ..., 0.0471, 0.9844, 0.7188],
         [0.5039, 0.4863, 0.3594,  ..., 0.3926, 0.4648, 0.3750],
         ...,
         [0.9180, 0.8750, 0.0942,  ..., 0.0732, 0.9609, 0.5898],
         [0.9258, 0.8984, 0.0859,  ..., 0.0698, 0.9609, 0.6133],
         [0.9258, 0.8828, 0.0894,  ..., 0.0688, 0.9648, 0.5938]],
        dtype=torch.bfloat16)}

A more powerful way of reconstructing the methylation values is using chain-of-thought. With additional test-time compute, we can let the model "think harder" about the problem, which can lead to better performance. However, it also takes considerably longer dependending on the number of thinking steps.

In [ ]:
quick_setup_pred_meth_cot = trainer.predict(
    model=model,
    datamodule=quick_setup_datamodule,
    predict_mode="reconstruct",
    genomic_locations=genomic_locations,
    species="homo_sapiens",
    n_thinking_steps=5,
    thinking_step_size=1000,
    uncertainty_quantile=0.1,
    return_keys=["pred_meth"],
)

cpgpt -CpGPTDataset: Initializing class CpGPTDataset.
cpgpt -CpGPTDataset: Loaded existing dataset metrics.


Output()

In [ ]:
quick_setup_pred_meth_cot["pred_meth"] = m_to_beta(quick_setup_pred_meth_cot["pred_meth"])
quick_setup_pred_meth_cot

{'pred_meth': tensor([[0.9414, 0.9023, 0.0732,  ..., 0.0544, 0.9805, 0.6719],
         [0.9453, 0.9102, 0.0796,  ..., 0.0564, 0.9844, 0.7148],
         [0.4570, 0.3809, 0.2148,  ..., 0.2314, 0.5273, 0.2334],
         ...,
         [0.9141, 0.8711, 0.0859,  ..., 0.0767, 0.9570, 0.5781],
         [0.9180, 0.8906, 0.0825,  ..., 0.0762, 0.9570, 0.6016],
         [0.9180, 0.8711, 0.0835,  ..., 0.0732, 0.9609, 0.5781]],
        dtype=torch.bfloat16)}

### 5.5 Analyze Attention Weights

The amount of memory required to store the attention weights is enormous. Therefore, we only use 1000 features for the demonstration. Also, remember that the the first token is the CLS token.

In [ ]:
quick_setup_attn = trainer.predict(
    model=model,
    datamodule=quick_setup_datamodule_attn,
    predict_mode="attention",
    aggregate_heads="mean",
    layer_index=-1,
    return_keys=["attention_weights", "chroms", "positions", "mask_na", "meth"],
)

cpgpt -CpGPTDataset: Initializing class CpGPTDataset.
cpgpt -CpGPTDataset: Loaded existing dataset metrics.


Output()

In [ ]:
quick_setup_attn

{'attention_weights': tensor([[[0.0013, 0.0011, 0.0012,  ...,    nan,    nan,    nan],
          [0.0012, 0.0013, 0.0013,  ...,    nan,    nan,    nan],
          [0.0012, 0.0013, 0.0014,  ...,    nan,    nan,    nan],
          ...,
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan],
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan],
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan]],
 
         [[0.0012, 0.0012, 0.0013,  ...,    nan,    nan,    nan],
          [0.0012, 0.0012, 0.0013,  ...,    nan,    nan,    nan],
          [0.0012, 0.0012, 0.0013,  ...,    nan,    nan,    nan],
          ...,
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan],
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan],
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan]],
 
         [[0.0134, 0.0122, 0.0129,  ...,    nan,    nan,    nan],
          [0.0125, 0.0126, 0.0110,  ...,    nan,    nan,    nan],
          [0.0126, 